In [34]:
import pandas as pd
import os
import numpy as np
from dateutil import parser
import logging
from datetime import datetime

In [35]:
def get_xlsx_data(directory):
    # 检查目录是否存在
    if not os.path.isdir(directory):
        print("提供的路径不是一个有效的目录")
        return

    # 初始化一个空的 DataFrame 来存储所有文件的数据
    all_data = pd.DataFrame()

    # 遍历目录及其子目录中的所有 Excel 文件
    for root, dirs, files in os.walk(directory):
        for filename in files:
            if filename.endswith('news.xlsx') or filename.endswith('市级整合.xlsx'):
                file_path = os.path.join(root, filename)
                print(f"获取 {filename} 成功")
                # 读取 Excel 文件
                df = pd.read_excel(file_path)

                if 'location' not in df.columns or df['location'].isnull().all():
                    # 增加省份名称
                    parts = filename.split('_')
                    df['location'] = parts[0]
                # 添加数据到总 DataFrame
                all_data = pd.concat([all_data, df], ignore_index=True)

    # 检查必要的列是否存在
    required_columns = ['location', 'title', 'subject', 'description', 'source_department',
                        'release_time', 'update_time', 'open_conditions', 'data_volume',
                        'is_api', 'file_type', 'access_count', 'download_count',
                        'api_call_count', 'link', 'update_cycle']
    if not all(col in all_data.columns for col in required_columns):
        print("某些必要列缺失")
        return

    return all_data

In [36]:

import time


def convert_data_volume(volume):
    """
    将不同格式的数据容量转换为统一的整数形式，单位为MB。

    参数:
    volume (str, float, or int): 原始数据容量字符串或数值，如 "1G", "0.5", "30M" 或数字。

    返回:
    int: 转换后的数据容量整数，若为 "若干" 则返回 1，无法转化为整数则返回 0。
    """
    try:
        if not volume:
            return 0  # 转换为0
        if isinstance(volume, str):  # 如果volume是字符串，执行单位转换
            volume = volume.strip()  # 去除字符串两端的空格
            if volume == '若干':
                return 1  # 转换为1
            if 'G' in volume:
                return int(float(volume.upper().replace('G', '')) * 1024)  # 转换为MB，转为整数
            elif 'M' in volume:
                return int(float(volume.upper().replace('M', '')))  # 去除M，转换为整数
            else:
                return int(float(volume))  # 没有单位，转换为浮点数后转为整数
        elif isinstance(volume, (float, int)):  # 检查volume是否为数值（包括float和int）
            return int(volume)  # 转换为整数
    except ValueError as e:
        return 0  # 返回0，避免在转换为整数时出现错误

    return 0  # 返回0，避免没有匹配到任何情况时的错误

# 将 'is_api' 字段转换为布尔值
def convert_to_bool(value):
    if isinstance(value, str):
        return value.lower() == 'true'
    return bool(value)

def fill_empty_subjects(df):
    """
    将 DataFrame 中 'subject' 列为空的值赋值为 '其他'

    参数:
    df (pd.DataFrame): 需要处理的 DataFrame

    返回:
    pd.DataFrame: 处理后的 DataFrame
    """
    # 使用 fillna 方法将 'subject' 列为空的值填充为 '其他'
    df['subject'] = df['subject'].fillna('其他')
    return df

def calculate_update_frequency(row):
    """
    根据更新周期计算每月更新频率。

    参数:
    row (Series): 包含了需要的更新信息的行数据。

    返回:
    float: 每月更新次数。
    """
    release_time = pd.to_datetime(row['release_time'], errors='coerce')
    update_time = pd.to_datetime(row['update_time'], errors='coerce')

    # 如果发布时间晚于更新时间，则不更新
    if release_time >= update_time:
        return 0

    update_cycle = row['update_cycle']

    # 如果更新周期不是字符串，直接返回0
    if not isinstance(update_cycle, str):
        return 0
    update_cycle = update_cycle.replace("‘", '').replace("’",'').strip()
    # 建立更新周期与每年更新次数的映射关系
    frequency_map = {
        '每年': 1, '按年': 1, '按年更新': 1, '1年': 1, '年': 1, '一年': 1, '年年': 1, '12个月': 1,
        '每半年': 2, '半年更新': 2, '6个月': 2, '六个月': 2,
        '每两年': 1/2,  '自定义：每两年': 1/2, '自定义：2年': 1/2,
        '每三年': 1/3, '自定义：每三年': 1/3, '三年': 1/3, '自定义：三年': 1/3,
        '每五年': 1/5,  '自定义：5年': 1/5,
        '每十年': 1/10, '自定义：10年': 1/10,
        '每季度': 4, '按季度': 4, '每季': 4, '按季': 4, '按季度更新': 4, '按季更新': 4, '季度': 4, '3个月': 4,
        '两个月': 12 / 2,
        '每月': 12, '按月': 12,  '按月更新': 12, '月': 12,
        '每周': 52, '按周': 52, '按周更新': 52,
        '实时': 365, '即时': 365, '每日': 365, '按日更新': 365, '每天': 365, '按天': 365, '实时更新': 365, '按天更新': 365,
        '按小时': 24 * 30, '小时级': 24 * 30,
        '按分钟': 60 * 24 * 30, '分钟级': 60 * 24 * 30,
        '其他': 0, '未定义': 0, '自定义': 0, '静态数据': 0, '静态': 0,
        '自定义：不更新': 0, '自定义：静态文件不更新': 0, '自定义：数据截至2020年': 0, '自定义：不在此平台更新': 0,
        '自定义：不在此平台发布更新': 0, '自定义：历史数据不更新': 0, '自定义：无需更新': 0, '自定义：无变化不更新': 0,
        '自定义：用户自定义': 0, '自定义：有变动时更新': 0, '自定义：不定期，有变动时更新': 0, '自定义：有变化时更新': 0, '自定义：不定期更新': 0, '自定义：按需更新': 0, '自定义：工业总量不对外': 0, '自定义：投资总量数据不对外': 0, '自定义：有更新的信息就更新': 0, '自定义：有新的类型再进行更新': 0, '自定义：人口普查工作开展时更': 0,
        '自定义：每次普查结果实时更新': 0, '自定义：申请维修支出时查询': 0, '自定义：有案件更新时更新数据': 0,
        '自定义：高院发布数据时更新': 0, '自定义：有普查更新时更新': 0, '自定义：普查时更新': 0, '自定义：有更新时更新': 0, '自定义：发生实际变更时': 0, '自定义：按批复文进行更新': 0, '自定义：根据实际情况': 0, '自定义：以后由应急管理局发': 0,'未知': 0, '适时': 0, '信用承诺': 0, '按需更新': 0,
    }

    # 若更新周期在映射字典中，则返回每月更新次数，否则返回0
    return frequency_map.get(update_cycle, 0) / 12

def calculate_is_update(row):
    """
    计算更新频率。

    参数:
    row (Series): DataFrame中的一行。

    返回:
    bool: 指示文本是否有更新。
    """
    release_time = pd.to_datetime(row['release_time'], errors='coerce')
    update_time = pd.to_datetime(row['update_time'], errors='coerce')

    # 如果发布时间晚于更新时间，则不更新
    if pd.notnull(release_time) and pd.notnull(update_time) and release_time < update_time:
        return True
    else:
        return False

def safe_parse_datetime(date_str):
    """尝试解析日期字符串，兼容多种格式，如果失败则返回 pd.NaT."""
    try:
        # dateutil.parser.parse 更加灵活，可以解析不完全一致的日期格式
        return parser.parse(date_str)
    except ValueError:
        return pd.NaT

def analyze_and_save_data(folder_path, df):
    """
    分析Excel数据，计算统计信息并保存至文件。

    参数:
    folder_path (str): 文件保存路径。
    df (DataFrame): 包含所有需要处理的数据的DataFrame。
    """
    # 处理data_volume列，将所有容量转换为MB
    df['data_volume'] = df['data_volume'].apply(convert_data_volume)
    logging.info("数据容量转换完成")

    # 计算平均每月更新频率
    df['平均更新/月'] = df.apply(calculate_update_frequency, axis=1).round(2)
    logging.info("平均更新频率计算完成")

    # 计算是否更新列
    df['is_update'] = df.apply(calculate_is_update, axis=1)
    logging.info("是否更新列计算完成")

    # 确保release_time和update_time列是字符串类型
    df['release_time'] = df['release_time'].astype(str)
    df['update_time'] = df['update_time'].astype(str)

    # 使用numpy.where选择release_time或update_time
    # 将 release_time 为空字符串或者为字符串 "nan" 的情况下，使用 update_time 来填充 chosen_time
    # 将 "nan" 替换为空字符串
    df['chosen_time'] = df['release_time'].fillna('').replace('nan', '')
    df['chosen_time'] = np.where(df['chosen_time'] == '', df['update_time'], df['chosen_time'])

    # 应用自定义的安全日期解析函数，并提取年份
    df['year'] = df['chosen_time'].apply(safe_parse_datetime).dt.year
    logging.info("年份列计算完成")

     # 将x映射为相应的开放条件
    open_conditions_mapping = {
    '无条件开放': '无条件开放',
    '无条件': '无条件开放',
    '登录开放': '无条件开放',
    '普遍开放': '无条件开放',
    '实名认证开放': '无条件开放',
    '普通开放': '无条件开放',
    '有条件': '有条件开放',
    '有条件开放': '有条件开放',
    '依申请开放': '有条件开放',
    '审核开放': '有条件开放',
    '有条件共享': '有条件开放',
    '实名注册用户可下载': '有条件开放',
    '申请开放': '有条件开放',
    '完全公开': '无条件开放',
    '部门审批': '有条件开放',
    '审批开放': '有条件开放',
    '依申请': '有条件开放',
    '依条件开放': '有条件开放',
    '根据业务需求提供': '有条件开放',
    '依申请公开': '有条件开放',
    '已申请开放': '有条件开放',
    '以申请开放': '有条件开放',
    '申请': '有条件开放',
    '需要用户授权': '有条件开放',
    '完全开放': '无条件开放',
    '半开放': '有条件开放',
    '不开放': '无条件开放',
    '开放': '无条件开放',
    '参考': '无条件开放',
    '需申请使用': '有条件开放',
    '依具体申请原因审核是否予以开放': '有条件开放',
    '脱敏开放': '有条件开放',
    '平台审核': '有条件开放',
    '申请审批': '有条件开放',
    '平台审批': '有条件开放',
    '需申请': '有条件开放',
    '工作参考': '无条件开放',
    '无条件开发': '无条件开放',
    '工作所需': '无条件开放',
    '申请后公开': '有条件开放',
    '申请后开放': '有条件开放',
    '无条件公开': '无条件开放',
    '用于数据核验': '无条件开放',
    '五条件共享': '有条件开放',  # 有疑问，请确认
    '无条件开房': '无条件开放',  # 有疑问，请确认
    '申请公开': '有条件开放',
    '需要提供查询函': '有条件开放',
    '行政依据': '无条件开放',
    '无条件共享，工作参考': '无条件开放',
    '须提供具体申请依据、实际用途和申请主体的相关信息。': '有条件开放',
    '眉山市有效外观明细表': '无条件开放',  # 有疑问，请确认
    '开放数据': '无条件开放',
    '社会保障卡启用': '无条件开放',  # 有疑问，请确认
    '办理施工许可证证可以申请查看': '有条件开放',
    '安全监督备案表': '无条件开放',
    '无条件开共享': '无条件开放',
    '此项政务信息资源仅用于数据校核、业务协同。': '无条件开放',
    '此数据仅限于工作参考和数据校核': '无条件开放',
    '本数据仅限用于工作参考': '无条件开放',
    '办公人员联络方式用于业务协调': '无条件开放',
    '可共享': '无条件开放',
    '无条件开放。': '无条件开放',
    '市级部门无条件共享': '无条件开放',
    '无条件共共享': '无条件开放',  # 有疑问，请确认
    '眉山市路灯照明工程情况': '无条件开放',  # 有疑问，请确认
    '有条件工程': '有条件开放',  # 有疑问，请确认
    '有': '有条件开放',
    '仅作工作参考': '无条件开放',
    '数据仅用于业务办理查询': '有条件开放',
    '数据仅用于业务办理': '有条件开放',
    '数据仅用于业务办理和查询': '有条件开放',
    '数据用于业务办理': '有条件开放',
    '不予共享': '无条件开放',
    '业务协同': '无条件开放',
    'WU': '无条件开放',  # 有疑问，请确认
    '仅用于工作参考': '无条件开放',
    '数据仅用于共享业务查询': '有条件开放',
    '农村计划生育家庭奖励扶助情况': '无条件开放',  # 有疑问，请确认
    '独生子女父母专项奖励情况': '无条件开放',  # 有疑问，请确认
    '共享': '无条件开放',
    '用作工作参考': '无条件开放',
    '依据：《中华人民共和国森林法》和《森林法实施条例》。': '无条件开放',  # 有疑问，请确认
    '提供查询的函。': '有条件开放',
    '用作业务协同': '无条件开放',
    '仅对上级法院系统开放': '有条件开放',
    '无条件g共享': '无条件开放',  # 有疑问，请确认
    '依申请共享': '有条件开放',
    '经申请': '有条件开放',
    '该数据仅供参考，如有疑问请联系眉山市公安局天府新区分局。': '有条件开放',  # 有疑问，请确认
    '公开数据': '无条件开放',
    '有条件开发': '有条件开放',
    '对所有人员公开': '无条件开放',
    '已申请': '有条件开放',
    '向社会开放': '无条件开放',
    '无条件开放共享': '无条件开放',
    '身份证申请或申请书': '有条件开放',
    '眉山天府新区林区分布一览表': '无条件开放',  # 有疑问，请确认
    '彭山区科普惠民共享基地名单': '无条件开放',  # 有疑问，请确认
    '区科协开展科技下乡活动统计': '无条件开放',  # 有疑问，请确认
    '彭山区农技协名单': '无条件开放',  # 有疑问，请确认
    '先申请': '有条件开放',
    '无条件共享类': '无条件开放',  # 有疑问，请确认
    '与残联相关业务': '无条件开放',  # 有疑问，请确认
    '无。': '无条件开放',  # 有疑问，请确认
    '青神县融媒体中心广播电视编辑记者采编考试合格证持证情况': '无条件开放',  # 有疑问，请确认
    '单位来函': '有条件开放',
    '单位发函': '有条件开放',
    '仅用于数据参考。': '无条件开放',
    '仅用于数据参考': '无条件开放',
    '仅用于工作参考。': '无条件开放',
    '仅作为工作参考': '无条件开放',
    '仅供工作参考': '无条件开放',
    '仅用于工作参考、数据校核。': '无条件开放',
    '仅用于工作参考，数据校核。': '无条件开放',
    '用于数据校核、工作参考、业务协同。': '无条件开放',
    '用于工作参考可申请共享': '有条件开放',
    '用于工作需要可申请共享': '有条件开放',
    '洪雅县外贸企业名单': '无条件开放',  # 有疑问，请确认
    '作为工作参考。': '无条件开放',
    '作为行政执法依据进行公开': '无条件开放',
    '行政公开': '无条件开放',
    '申请同意后开放': '有条件开放',
    '无条件分享': '无条件开放',
    '有条件分享': '有条件开放',
    '文件': '无条件开放',
    '无条件开放数据': '无条件开放',
    '数据仅用于业务查询办理': '有条件开放',
    '向社会公开': '无条件开放',
    '对所有人公开': '无条件开放',
    '审计法治工作报告': '无条件开开放',
    '无条件是使用，用于开展正常工作': '无条件开放',
    '仅用于工作参考，业务协同。': '无条件开放',
    '洪雅县地方政府债务限额汇总表': '无条件开放',  # 有疑问，请确认
    '公务用车购置及运行费统计表': '无条件开放',  # 有疑问，请确认
    '有条件公共享': '有条件开放',  # 有疑问，请确认
    '彭山称号企业汇总': '无条件开放',  # 有疑问，请确认
    '无条件共享，作为政务信息公开。': '无条件开放',
    '作为行政依据、业务参考': '无条件开放',
    '眉山市市级示范社名录': '无条件开放',  # 有疑问，请确认
    '政务信息主动公开流程图': '无条件开放',  # 有疑问，请确认
    '个体工商户变更登记流程图': '无条件开放',  # 有疑问，请确认
    '个体工商户开业登记流程图': '无条件开放',  # 有疑问，请确认
    '个体工商户注销登记流程图': '无条件开放',  # 有疑问，请确认
    '名称争议事项流程图': '无条件开放',  # 有疑问，请确认
    '有业务需求的进行共享': '有条件开放',
    '信访条例': '无条件开放',
    '预算法': '无条件开放',
    '无条件贡献': '无条件开放',
    '作为行政依据，工作参考。': '无条件开放',
    '龙马镇商贸行业燃气安全隐患管理台账': '无条件开放',  # 有疑问，请确认
    '依条件申请': '有条件开放',
    '重点服务业企业培育情况表': '无条件开放',  # 有疑问，请确认
    '需向部门申请': '有条件开放',
    '政务信息资源': '无条件开放',
    '区科协科普e展点位': '无条件开放',  # 有疑问，请确认
    '彭山区农技协开展培训情况': '无条件开放',  # 有疑问，请确认
    '需要可自行下载': '无条件开放',
    '开发共享下载': '无条件开放',
    '开发下载': '无条件开放',
    '开放下载': '无条件开放',
    '可作为工作参考': '无条件开放',
    '需求部门申请共享': '有条件开放',
    '无条件分析': '无条件开放',
    '廉政法规知识测试资料': '无条件开放',
    '丹棱县村级小微权力清单': '无条件开放',  # 有疑问，请确认
    '丹棱县纪委年度部门预算': '无条件开放',  # 有疑问，请确认
    '群众信访举报途径电子表格数据': '无条件开放',  # 有疑问，请确认
    '用于档案数据校验': '无条件开放',
    '地方政府专项债券分区域限额': '无条件开放',  # 有疑问，请确认
    '用于档案查阅利用': '无条件开放',
    '申请同意后开放共享': '有条件开放',
    '眉山市公共资源交易领域信息主动公开目录': '无条件开放',  # 有疑问，请确认
    '需要可以出函申请': '有条件开放',
    '共享平台': '无条件开放',
    '需个人或单位出具查档介绍信或查档函': '有条件开放',
    '获奖名单': '无条件开放',  # 有疑问，请确认
    '招生依据': '无条件开放',  # 有疑问，请确认
    '用于数据校核': '无条件开放',
    '非盈利性使用': '无条件开放',
    '无条件开发1': '无条件开放',  # 有疑问，请确认
    '查询人基本信息，身份证信申请查询': '有条件开放',
    '待县人大批准2020年部门决算后开放': '有条件开放',
    '县人大批准2020年部门决算后开放': '有条件开放',
    '无条件共享开放': '无条件开放',
    '眉山司法局香港、澳门永久性居民中的中国居民申请在眉从事律师职业核准办事指南': '无条件开放',  # 有疑问，请确认
    '中华人民共和国公证法': '无条件开放',
    '眉山市司法局兼职律师执业许可办事指南': '无条件开放',  # 有疑问，请确认
    '眉山市司法局律师事务所设立办事指南': '无条件开放',  # 有疑问，请确认
    '眉山市司法局关于设立律师事务所（分所）的指南': '无条件开放',  # 有疑问，请确认
    '眉山市司法局法律援助律师执业许可办事指南': '无条件开放',  # 有疑问，请确认
    '眉山市司法局台湾居民申请在川从事律师职业许可办事指南': '无条件开放',  # 有疑问，请确认
    '眉山市司法局专职律师执业许可办事指南': '无条件开放',  # 有疑问，请确认
    '眉山公证机构及公证员基本信息': '无条件开放',  # 有疑问，请确认
    '共享类文件': '无条件开放',
    '农村居民可支配收入与支出': '无条件开放',  # 有疑问，请确认
    '2020年丹棱县本级一般公共预算支出预算表': '无条件开放',  # 有疑问，请确认
    '2020年丹棱县政府性基金收入预算表': '无条件开放',  # 有疑问，请确认
    '填写申请后开放': '有条件开放',
    '有条件共享（申请）': '有条件开放',
    '登录公开': '无条件开放',
    '按相关规定审核开放共享': '有条件开放',
    '按需审批使用': '有条件开放',
    '按需审核使用': '有条件开放',
    '用于数据校验': '无条件开放',
    '垃圾分类等场景': '无条件开放',  # 有疑问，请确认
    '按相关政策要求，当前数据允许开放': '无条件开放',
    '招聘会信息，无条件开放': '无条件开放',
    '无条件开': '无条件开放',
    '受限开放': '有条件开放',
    '用于数据查看与应用': '无条件开放',
    '因城市管理、综合执法需要可使用本数据': '无条件开放',
    '按需开放': '有条件开放',
    '开放网站': '无条件开放',
    '申请共享': '有条件开放',
    '开放网站申请': '有条件开放',
    '根据接口规范': '无条件开放',
    '1级\t不脱敏': '无条件开放',  # 有疑问，请确认
    '依据申请': '有条件开放'
}

    df['open_conditions'] = df['open_conditions'].map(open_conditions_mapping)
    logging.info('open_conditions 列已映射完成')

    # 分组统计各项数据
    grouped = df.groupby(['location', 'source_department', 'subject', 'year'])
    summary = grouped.agg({
        'title': 'count',  # 文本数
        'data_volume': lambda x: sum(float(volume.replace('M', '')) for volume in x),  # 总数据量(MB)
        '平均更新/月': 'mean',  # 平均每月更新频率
        'is_update': 'mean',  # 更新文本率
        'open_conditions': [
            lambda x: (x == '无条件开放').mean(),  # 无条件开放文本率
            lambda x: (x == '有条件开放').mean()  # 有条件开放文本率
        ],
        'is_api': lambda x: x.sum() / len(x),  # 有接口文本率
        'access_count': 'sum'  # 总访问次数
    }).reset_index()

    # 重命名统计列
    summary.columns = ['地区', '来源部门', '主题', '年份', '文本数', '总数据量(MB)', '平均更新/月', '更新文本率', '无条件开放文本率', '有条件开放文本率', '有接口文本率', '总访问次数']

    # 保存统计信息到Excel文件
    summary.to_excel(f'{folder_path}/统计信息汇总.xlsx', index=False)

    print("统计信息已保存至:", f'{folder_path}/统计信息汇总_{time.strftime("%Y%m%d_%H%M%S")}.xlsx')


In [37]:
# 使用示例
folder_path = r'E:\pythonProject\outsource\China_province_opened_data\output'
df = get_xlsx_data(folder_path)

获取 安徽省_开放数据_市级整合.xlsx 成功
获取 北京市_news.xlsx 成功
获取 重庆市_news.xlsx 成功
获取 三明市_api_news.xlsx 成功
获取 三明市_news.xlsx 成功
获取 南平市_api_news.xlsx 成功
获取 南平市_news.xlsx 成功
获取 厦门市_api_news.xlsx 成功
获取 厦门市_news.xlsx 成功
获取 宁德市_api_news.xlsx 成功
获取 宁德市_news.xlsx 成功
获取 平潭综合实验区_api_news.xlsx 成功
获取 平潭综合实验区_news.xlsx 成功
获取 泉州市_api_news.xlsx 成功
获取 泉州市_news.xlsx 成功
获取 漳州市_api_news.xlsx 成功
获取 漳州市_news.xlsx 成功
获取 福州市_api_news.xlsx 成功
获取 福州市_news.xlsx 成功
获取 莆田市_api_news.xlsx 成功
获取 莆田市_news.xlsx 成功
获取 龙岩市_api_news.xlsx 成功
获取 龙岩市_news.xlsx 成功
获取 东莞市_news.xlsx 成功
获取 中山市_news.xlsx 成功
获取 云浮市_news.xlsx 成功
获取 佛山市_news.xlsx 成功
获取 广州市_news.xlsx 成功
获取 惠州市_news.xlsx 成功
获取 揭阳市_news.xlsx 成功
获取 梅州市_news.xlsx 成功
获取 汕头市_news.xlsx 成功
获取 汕尾市_news.xlsx 成功
获取 江门市_news.xlsx 成功
获取 河源市_news.xlsx 成功
获取 深圳市_news.xlsx 成功
获取 清远市_news.xlsx 成功
获取 湛江市_news.xlsx 成功
获取 潮州市_news.xlsx 成功
获取 珠海市_news.xlsx 成功
获取 肇庆市_news.xlsx 成功
获取 茂名市_news.xlsx 成功
获取 阳江市_news.xlsx 成功
获取 韶关市_news.xlsx 成功
获取 广西省开放数据_市级整合.xlsx 成功
获取 六盘水市_news.xlsx 成功
获取 安顺市_news.xlsx 成功
获取

In [38]:
df

,province,location,title,subject,description,source_department,release_time,update_time,open_conditions,data_volume,is_api,file_type,access_count,download_count,api_call_count,link,update_cycle
0,安徽,蚌埠市,蚌埠市2024年市级惠民惠农补贴政策清单,民生服务，金融财税，,NaN,蚌埠市市财政局,2024-05-31,2024-05-31,NaN,0,False,['XLS'],10.0,1.0,0.0,https://www.bengbu.gov.cn/site/tpl/6621?resour...,NaN
1,安徽,蚌埠市,蚌埠市发改委公共服务指南（2023年版）,工业商贸，政策法规，民生服务，信用服务，机构团体，公共安全，,NaN,蚌埠市市发展改革委,2024-05-31,2024-05-31,NaN,0,False,['XLS'],18.0,2.0,0.0,https://www.bengbu.gov.cn/site/tpl/6621?resour...,NaN
2,安徽,蚌埠市,蚌埠市自然资源和规划局（市林业局）2024年度随机抽查事项清单,政策法规，生态资源，,NaN,蚌埠市市自然资源和规划局,2024-05-31,2024-05-31,NaN,0,False,['XLS'],15.0,0.0,0.0,https://www.bengbu.gov.cn/site/tpl/6621?resour...,NaN
3,安徽,蚌埠市,蚌埠市流感疫苗接种点信息汇总,医疗卫生，公共安全，,NaN,蚌埠市市卫生健康委,2024-05-29,2024-05-29,NaN,0,False,['XLS'],27.0,1.0,0.0,https://www.bengbu.gov.cn/site/tpl/6621?resour...,NaN
4,安徽,蚌埠市,蚌埠市社保经办机构信息汇总表,社保就业，机构团体，,NaN,蚌埠市市人力资源社会保障局,2024-05-29,2024-05-29,NaN,0,False,['XLS'],17.0,1.0,0.0,https://www.bengbu.gov.cn/site/tpl/6621?resour...,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
428429,NaN,金华市,婺城区应急指挥-应急物资名录信息,资源能源,"婺城区应急指挥-应急物资名录信息,包含主键,物品编码,物资名称,物资标准名称ID,品牌ID等信息",婺城区应急管理局,2024-04-28,2024-06-21,申请开放,486,True,['api'],908.0,0.0,0.0,https://open.data.jinhua.gov.cn/jdop_front/det...,每月
428430,NaN,金华市,婺城区应急指挥-应急物资仓储信息,资源能源,"婺城区应急指挥-应急物资仓储信息,包含主键,安置场所ID,物资类型,物资名录ID,标准物资名...",婺城区应急管理局,2024-04-28,2024-06-21,申请开放,4824,True,['api'],907.0,0.0,0.0,https://open.data.jinhua.gov.cn/jdop_front/det...,每月
428431,NaN,金华市,金华市义乌市“一网通管”行政执法监管-附属客体特种设备信息,市场监督,(市级待审核)xzzf_issue_supervision_fs_equipment-附属客...,义乌市市场监管局,2023-04-24,2024-06-21,申请开放,4682,True,['api'],905.0,0.0,0.0,https://open.data.jinhua.gov.cn/jdop_front/det...,每年
428432,NaN,金华市,金华市义乌市“一网通管”行政执法监管-附属客体特定产品信息,市场监督,(市级待审核)xzzf_issue_supervision_fs_product-附属客体-...,义乌市市场监管局,2023-04-24,2024-06-21,申请开放,2355,True,['api'],905.0,0.0,0.0,https://open.data.jinhua.gov.cn/jdop_front/det...,每年


In [39]:
# df.to_csv('./test.csv')

In [40]:
# 获取 update_cycle 列的所有唯一值
unique_update_cycle = df['update_cycle'].unique()

unique_update_cycle

array([nan, '实时', '每年', '每月', '不定期更新', '每两年', '按年更新', '牲畜饲养', '按年', '每季度',
       '每天', '不定期', '每半年', '每日', '按小时', '按天', '按季度', '按月', '不更新', '按周',
       '按分钟', '每五年', '每三年', '每周', '其它', '按季更新', '不定时', '按周更新', '按日更新',
       '实时更新', '按月更新', '半年更新', '按日', '半年', '按季', '静态', '自定义：用户自定义',
       '自定义：每三年', '自定义：不定期', '自定义：有变动时更新', '自定义：不定期，有变动时更新',
       '自定义：数据截至2020年', '自定义：有变化时更新', '自定义：不定期更新', '自定义：每两年', '自定义：按需更新',
       '自定义：工业总量不对外', '自定义：投资总量数据不对外', '自定义：', '自定义：10年', '自定义：有更新的信息就更新',
       '自定义：3年', '自定义：5年', '自定义：三年', '自定义：有新的类型再进行更新', '自定义：人口普查工作开展时更',
       '自定义：每次普查结果实时更新', '自定义：申请维修支出时查询', '自定义：有案件更新时更新数据',
       '自定义：高院发布数据时更新', '自定义：有普查更新时更新', '自定义：普查时更新', '自定义：有更新时更新',
       '自定义：发生实际变更时', '自定义：按批复文进行更新', '自定义：根据实际情况', '自定义：以后由应急管理局发',
       '自定义：2年', '其他', '四个月', '6个月', '六个月', '12个月', '1年', '两个月', '按需更新',
       '按季度更新', '按天更新', '未知', '月', '年', '季度', '07', '信用承诺', '姣忔湀', '瀹炴椂',
       '姣忓勾', '未定义', '自定义', '静态数据', '即时', '每季', '一年', '三年', '适时', '3个月',
       '年年', '小

In [41]:
# 获取 update_cycle 列的所有唯一值
unique_update_cycle = df['open_conditions'].unique()

unique_update_cycle

array([nan, '普通公开，简单注册用户可下载', '普通公开,简单注册用户可下载', '实名注册用户可下载', '无条件开放',
       '有条件开放', '无条件', '无条件共享', '无', '无条件更新', '登录开放', '申请开放', '完全公开',
       '依申请开放', '有条件共享', '有条件', "('无条件开放',)", "('有条件开放',)", '有条件公开',
       '有开放条件', '一般公开', '普通开放', '未知', '部门审批', '审批开放', '依申请', '依条件开放',
       '根据业务需求提供', '依申请公开', '已申请开放', '以申请开放', '依申请开放。根据管理职责依申请对行业管理部门开放。',
       '申请', '需要用户授权', '完全开放', '半开放', '不开放', '开放', '参考', '需申请使用',
       '依具体申请原因审核是否予以开放', '暂无', '脱敏开放', '平台审核', '申请审批', '平台审批', '需申请',
       '工作参考', '无条件开发', '工作所需', '东坡区', '申请后公开', '申请后开放', '无条件公开',
       '用于数据核验', '有条件共享。', '五条件共享', '无条件开房', '申请公开', '需要提供查询函', '行政依据',
       '无条件共享，工作参考', '须提供具体申请依据、实际用途和申请主体的相关信息。', '眉山市有效外观明细表', '开放数据',
       '社会保障卡启用', '办理施工许可证证可以申请查看', '安全监督备案表', '无条件开共享',
       '此项政务信息资源仅用于数据校核、业务协同。', '此数据用于工作参考', '此数据只作为工作参考',
       '该数据仅限于工作参考和数据校核', '本数据仅限用于工作参考', '办公人员联络方式用于业务协调', '可共享',
       '无条件开放。', '市级部门无条件共享', '无条件共共享', '眉山市路灯照明工程情况', '有条件工程', '有', '1',
       '作为行政依据、工作参考', '仅作工作参考', '数据仅用于业务办理查

In [42]:
# df_copy = df.copy()
# analyze_and_save_data(folder_path, df_copy)

In [43]:
df_copy = df.copy()

df_copy['data_volume'] = df_copy['data_volume'].apply(convert_data_volume)
print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')} - 数据容量转换完成")

# 计算平均每月更新频率
df_copy['平均更新/月'] = df_copy.apply(calculate_update_frequency, axis=1).round(2)
print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')} - 平均更新频率计算完成")

# 计算是否更新列
df_copy['is_update'] = df_copy.apply(calculate_is_update, axis=1)
print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')} - 是否更新列计算完成")

# 确保release_time和update_time列是字符串类型
df_copy['release_time'] = df_copy['release_time'].astype(str)
df_copy['update_time'] = df_copy['update_time'].astype(str)

# 使用numpy.where选择release_time或update_time
# 将 release_time 为空字符串或者为字符串 "nan" 的情况下，使用 update_time 来填充 chosen_time
# 将 "nan" 替换为空字符串
df_copy['chosen_time'] = df_copy['release_time'].fillna('').replace('nan', '')
df_copy['chosen_time'] = np.where(df_copy['chosen_time'] == '', df_copy['update_time'], df_copy['chosen_time'])

# 应用自定义的安全日期解析函数，并提取年份
df_copy['year'] = df_copy['chosen_time'].apply(safe_parse_datetime).dt.year
print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')} - 年份列计算完成")

 # 将x映射为相应的开放条件
open_conditions_mapping = {
'无条件开放': '无条件开放',
'无条件': '无条件开放',
'登录开放': '无条件开放',
'普遍开放': '无条件开放',
'实名认证开放': '无条件开放',
'普通开放': '无条件开放',
'有条件': '有条件开放',
'有条件开放': '有条件开放',
'依申请开放': '有条件开放',
'审核开放': '有条件开放',
'有条件共享': '有条件开放',
'实名注册用户可下载': '有条件开放',
'申请开放': '有条件开放',
'完全公开': '无条件开放',
'部门审批': '有条件开放',
'审批开放': '有条件开放',
'依申请': '有条件开放',
'依条件开放': '有条件开放',
'根据业务需求提供': '有条件开放',
'依申请公开': '有条件开放',
'已申请开放': '有条件开放',
'以申请开放': '有条件开放',
'申请': '有条件开放',
'需要用户授权': '有条件开放',
'完全开放': '无条件开放',
'半开放': '有条件开放',
'不开放': '无条件开放',
'开放': '无条件开放',
'参考': '无条件开放',
'需申请使用': '有条件开放',
'依具体申请原因审核是否予以开放': '有条件开放',
'脱敏开放': '有条件开放',
'平台审核': '有条件开放',
'申请审批': '有条件开放',
'平台审批': '有条件开放',
'需申请': '有条件开放',
'工作参考': '无条件开放',
'无条件开发': '无条件开放',
'工作所需': '无条件开放',
'申请后公开': '有条件开放',
'申请后开放': '有条件开放',
'无条件公开': '无条件开放',
'用于数据核验': '无条件开放',
'五条件共享': '有条件开放',  # 有疑问，请确认
'无条件开房': '无条件开放',  # 有疑问，请确认
'申请公开': '有条件开放',
'需要提供查询函': '有条件开放',
'行政依据': '无条件开放',
'无条件共享，工作参考': '无条件开放',
'须提供具体申请依据、实际用途和申请主体的相关信息。': '有条件开放',
'眉山市有效外观明细表': '无条件开放',  # 有疑问，请确认
'开放数据': '无条件开放',
'社会保障卡启用': '无条件开放',  # 有疑问，请确认
'办理施工许可证证可以申请查看': '有条件开放',
'安全监督备案表': '无条件开放',
'无条件开共享': '无条件开放',
'此项政务信息资源仅用于数据校核、业务协同。': '无条件开放',
'此数据仅限于工作参考和数据校核': '无条件开放',
'本数据仅限用于工作参考': '无条件开放',
'办公人员联络方式用于业务协调': '无条件开放',
'可共享': '无条件开放',
'无条件开放。': '无条件开放',
'市级部门无条件共享': '无条件开放',
'无条件共共享': '无条件开放',  # 有疑问，请确认
'眉山市路灯照明工程情况': '无条件开放',  # 有疑问，请确认
'有条件工程': '有条件开放',  # 有疑问，请确认
'有': '有条件开放',
'仅作工作参考': '无条件开放',
'数据仅用于业务办理查询': '有条件开放',
'数据仅用于业务办理': '有条件开放',
'数据仅用于业务办理和查询': '有条件开放',
'数据用于业务办理': '有条件开放',
'不予共享': '无条件开放',
'业务协同': '无条件开放',
'WU': '无条件开放',  # 有疑问，请确认
'仅用于工作参考': '无条件开放',
'数据仅用于共享业务查询': '有条件开放',
'农村计划生育家庭奖励扶助情况': '无条件开放',  # 有疑问，请确认
'独生子女父母专项奖励情况': '无条件开放',  # 有疑问，请确认
'共享': '无条件开放',
'用作工作参考': '无条件开放',
'依据：《中华人民共和国森林法》和《森林法实施条例》。': '无条件开放',  # 有疑问，请确认
'提供查询的函。': '有条件开放',
'用作业务协同': '无条件开放',
'仅对上级法院系统开放': '有条件开放',
'无条件g共享': '无条件开放',  # 有疑问，请确认
'依申请共享': '有条件开放',
'经申请': '有条件开放',
'该数据仅供参考，如有疑问请联系眉山市公安局天府新区分局。': '有条件开放',  # 有疑问，请确认
'公开数据': '无条件开放',
'有条件开发': '有条件开放',
'对所有人员公开': '无条件开放',
'已申请': '有条件开放',
'向社会开放': '无条件开放',
'无条件开放共享': '无条件开放',
'身份证申请或申请书': '有条件开放',
'眉山天府新区林区分布一览表': '无条件开放',  # 有疑问，请确认
'彭山区科普惠民共享基地名单': '无条件开放',  # 有疑问，请确认
'区科协开展科技下乡活动统计': '无条件开放',  # 有疑问，请确认
'彭山区农技协名单': '无条件开放',  # 有疑问，请确认
'先申请': '有条件开放',
'无条件共享类': '无条件开放',  # 有疑问，请确认
'与残联相关业务': '无条件开放',  # 有疑问，请确认
'无。': '无条件开放',  # 有疑问，请确认
'青神县融媒体中心广播电视编辑记者采编考试合格证持证情况': '无条件开放',  # 有疑问，请确认
'单位来函': '有条件开放',
'单位发函': '有条件开放',
'仅用于数据参考。': '无条件开放',
'仅用于数据参考': '无条件开放',
'仅用于工作参考。': '无条件开放',
'仅作为工作参考': '无条件开放',
'仅供工作参考': '无条件开放',
'仅用于工作参考、数据校核。': '无条件开放',
'仅用于工作参考，数据校核。': '无条件开放',
'用于数据校核、工作参考、业务协同。': '无条件开放',
'用于工作参考可申请共享': '有条件开放',
'用于工作需要可申请共享': '有条件开放',
'洪雅县外贸企业名单': '无条件开放',  # 有疑问，请确认
'作为工作参考。': '无条件开放',
'作为行政执法依据进行公开': '无条件开放',
'行政公开': '无条件开放',
'申请同意后开放': '有条件开放',
'无条件分享': '无条件开放',
'有条件分享': '有条件开放',
'文件': '无条件开放',
'无条件开放数据': '无条件开放',
'数据仅用于业务查询办理': '有条件开放',
'向社会公开': '无条件开放',
'对所有人公开': '无条件开放',
'审计法治工作报告': '无条件开开放',
'无条件是使用，用于开展正常工作': '无条件开放',
'仅用于工作参考，业务协同。': '无条件开放',
'洪雅县地方政府债务限额汇总表': '无条件开放',  # 有疑问，请确认
'公务用车购置及运行费统计表': '无条件开放',  # 有疑问，请确认
'有条件公共享': '有条件开放',  # 有疑问，请确认
'彭山称号企业汇总': '无条件开放',  # 有疑问，请确认
'无条件共享，作为政务信息公开。': '无条件开放',
'作为行政依据、业务参考': '无条件开放',
'眉山市市级示范社名录': '无条件开放',  # 有疑问，请确认
'政务信息主动公开流程图': '无条件开放',  # 有疑问，请确认
'个体工商户变更登记流程图': '无条件开放',  # 有疑问，请确认
'个体工商户开业登记流程图': '无条件开放',  # 有疑问，请确认
'个体工商户注销登记流程图': '无条件开放',  # 有疑问，请确认
'名称争议事项流程图': '无条件开放',  # 有疑问，请确认
'有业务需求的进行共享': '有条件开放',
'信访条例': '无条件开放',
'预算法': '无条件开放',
'无条件贡献': '无条件开放',
'作为行政依据，工作参考。': '无条件开放',
'龙马镇商贸行业燃气安全隐患管理台账': '无条件开放',  # 有疑问，请确认
'依条件申请': '有条件开放',
'重点服务业企业培育情况表': '无条件开放',  # 有疑问，请确认
'需向部门申请': '有条件开放',
'政务信息资源': '无条件开放',
'区科协科普e展点位': '无条件开放',  # 有疑问，请确认
'彭山区农技协开展培训情况': '无条件开放',  # 有疑问，请确认
'需要可自行下载': '无条件开放',
'开发共享下载': '无条件开放',
'开发下载': '无条件开放',
'开放下载': '无条件开放',
'可作为工作参考': '无条件开放',
'需求部门申请共享': '有条件开放',
'无条件分析': '无条件开放',
'廉政法规知识测试资料': '无条件开放',
'丹棱县村级小微权力清单': '无条件开放',  # 有疑问，请确认
'丹棱县纪委年度部门预算': '无条件开放',  # 有疑问，请确认
'群众信访举报途径电子表格数据': '无条件开放',  # 有疑问，请确认
'用于档案数据校验': '无条件开放',
'地方政府专项债券分区域限额': '无条件开放',  # 有疑问，请确认
'用于档案查阅利用': '无条件开放',
'申请同意后开放共享': '有条件开放',
'眉山市公共资源交易领域信息主动公开目录': '无条件开放',  # 有疑问，请确认
'需要可以出函申请': '有条件开放',
'共享平台': '无条件开放',
'需个人或单位出具查档介绍信或查档函': '有条件开放',
'获奖名单': '无条件开放',  # 有疑问，请确认
'招生依据': '无条件开放',  # 有疑问，请确认
'用于数据校核': '无条件开放',
'非盈利性使用': '无条件开放',
'无条件开发1': '无条件开放',  # 有疑问，请确认
'查询人基本信息，身份证信申请查询': '有条件开放',
'待县人大批准2020年部门决算后开放': '有条件开放',
'县人大批准2020年部门决算后开放': '有条件开放',
'无条件共享开放': '无条件开放',
'眉山司法局香港、澳门永久性居民中的中国居民申请在眉从事律师职业核准办事指南': '无条件开放',  # 有疑问，请确认
'中华人民共和国公证法': '无条件开放',
'眉山市司法局兼职律师执业许可办事指南': '无条件开放',  # 有疑问，请确认
'眉山市司法局律师事务所设立办事指南': '无条件开放',  # 有疑问，请确认
'眉山市司法局关于设立律师事务所（分所）的指南': '无条件开放',  # 有疑问，请确认
'眉山市司法局法律援助律师执业许可办事指南': '无条件开放',  # 有疑问，请确认
'眉山市司法局台湾居民申请在川从事律师职业许可办事指南': '无条件开放',  # 有疑问，请确认
'眉山市司法局专职律师执业许可办事指南': '无条件开放',  # 有疑问，请确认
'眉山公证机构及公证员基本信息': '无条件开放',  # 有疑问，请确认
'共享类文件': '无条件开放',
'农村居民可支配收入与支出': '无条件开放',  # 有疑问，请确认
'2020年丹棱县本级一般公共预算支出预算表': '无条件开放',  # 有疑问，请确认
'2020年丹棱县政府性基金收入预算表': '无条件开放',  # 有疑问，请确认
'填写申请后开放': '有条件开放',
'有条件共享（申请）': '有条件开放',
'登录公开': '无条件开放',
'按相关规定审核开放共享': '有条件开放',
'按需审批使用': '有条件开放',
'按需审核使用': '有条件开放',
'用于数据校验': '无条件开放',
'垃圾分类等场景': '无条件开放',  # 有疑问，请确认
'按相关政策要求，当前数据允许开放': '无条件开放',
'招聘会信息，无条件开放': '无条件开放',
'无条件开': '无条件开放',
'受限开放': '有条件开放',
'用于数据查看与应用': '无条件开放',
'因城市管理、综合执法需要可使用本数据': '无条件开放',
'按需开放': '有条件开放',
'开放网站': '无条件开放',
'申请共享': '有条件开放',
'开放网站申请': '有条件开放',
'根据接口规范': '无条件开放',
'1级\t不脱敏': '无条件开放',  # 有疑问，请确认
'依据申请': '有条件开放'
}

df_copy['open_conditions'] = df_copy['open_conditions'].map(open_conditions_mapping)
print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')} - open_conditions 列已映射完成")

df_copy['is_api'] = df_copy['is_api'].apply(convert_to_bool)
print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')} - is_api 列已转换为布尔值")

df_copy['subject'] = df_copy['subject'].fillna('其他')
print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')} - 'subject' 列空值填充完成")

2024-06-28 00:01:04 - 数据容量转换完成
2024-06-28 00:06:38 - 平均更新频率计算完成
2024-06-28 00:12:28 - 是否更新列计算完成
2024-06-28 00:12:44 - 年份列计算完成
2024-06-28 00:12:45 - open_conditions 列已映射完成
2024-06-28 00:12:45 - is_api 列已转换为布尔值
2024-06-28 00:12:45 - 'subject' 列空值填充完成


In [44]:
# df_copy.to_csv('./copy_test.csv')

In [45]:
# 分组统计各项数据
grouped = df_copy.groupby(['location', 'source_department', 'subject', 'year'])
summary = grouped.agg({
    'title': 'count',  # 文本数
    'data_volume': lambda x: sum(volume for volume in x),  # 总数据量(MB)
    '平均更新/月': 'mean',  # 平均每月更新频率
    'is_update': 'mean',  # 更新文本率
    'open_conditions': [
        lambda x: (x == '无条件开放').mean(),  # 无条件开放文本率
        lambda x: (x == '有条件开放').mean()  # 有条件开放文本率
    ],
    'is_api': 'mean',  # 有接口文本率
    'access_count': 'sum'  # 总访问次数
}).reset_index()

# 重命名统计列
summary.columns = ['地区', '来源部门', '主题', '年份', '文本数', '总数据量(MB)', '平均更新/月', '更新文本率', '无条件开放文本率', '有条件开放文本率', '有接口文本率', '总访问次数']

# 保存统计信息到Excel文件
summary.to_excel(f'{folder_path}/统计信息汇总_{time.strftime("%Y%m%d_%H%M%S")}.xlsx', index=False)

print("统计信息已保存至:", f'{folder_path}/统计信息汇总_{time.strftime("%Y%m%d_%H%M%S")}.xlsx')


统计信息已保存至: E:\pythonProject\outsource\China_province_opened_data\output/统计信息汇总_20240628_001312.xlsx


In [46]:
df_copy

,province,location,title,subject,description,source_department,release_time,update_time,open_conditions,data_volume,...,file_type,access_count,download_count,api_call_count,link,update_cycle,平均更新/月,is_update,chosen_time,year
0,安徽,蚌埠市,蚌埠市2024年市级惠民惠农补贴政策清单,民生服务，金融财税，,NaN,蚌埠市市财政局,2024-05-31,2024-05-31,NaN,0,...,['XLS'],10.0,1.0,0.0,https://www.bengbu.gov.cn/site/tpl/6621?resour...,NaN,0.00,False,2024-05-31,2024.0
1,安徽,蚌埠市,蚌埠市发改委公共服务指南（2023年版）,工业商贸，政策法规，民生服务，信用服务，机构团体，公共安全，,NaN,蚌埠市市发展改革委,2024-05-31,2024-05-31,NaN,0,...,['XLS'],18.0,2.0,0.0,https://www.bengbu.gov.cn/site/tpl/6621?resour...,NaN,0.00,False,2024-05-31,2024.0
2,安徽,蚌埠市,蚌埠市自然资源和规划局（市林业局）2024年度随机抽查事项清单,政策法规，生态资源，,NaN,蚌埠市市自然资源和规划局,2024-05-31,2024-05-31,NaN,0,...,['XLS'],15.0,0.0,0.0,https://www.bengbu.gov.cn/site/tpl/6621?resour...,NaN,0.00,False,2024-05-31,2024.0
3,安徽,蚌埠市,蚌埠市流感疫苗接种点信息汇总,医疗卫生，公共安全，,NaN,蚌埠市市卫生健康委,2024-05-29,2024-05-29,NaN,0,...,['XLS'],27.0,1.0,0.0,https://www.bengbu.gov.cn/site/tpl/6621?resour...,NaN,0.00,False,2024-05-29,2024.0
4,安徽,蚌埠市,蚌埠市社保经办机构信息汇总表,社保就业，机构团体，,NaN,蚌埠市市人力资源社会保障局,2024-05-29,2024-05-29,NaN,0,...,['XLS'],17.0,1.0,0.0,https://www.bengbu.gov.cn/site/tpl/6621?resour...,NaN,0.00,False,2024-05-29,2024.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
428429,NaN,金华市,婺城区应急指挥-应急物资名录信息,资源能源,"婺城区应急指挥-应急物资名录信息,包含主键,物品编码,物资名称,物资标准名称ID,品牌ID等信息",婺城区应急管理局,2024-04-28,2024-06-21,有条件开放,486,...,['api'],908.0,0.0,0.0,https://open.data.jinhua.gov.cn/jdop_front/det...,每月,1.00,True,2024-04-28,2024.0
428430,NaN,金华市,婺城区应急指挥-应急物资仓储信息,资源能源,"婺城区应急指挥-应急物资仓储信息,包含主键,安置场所ID,物资类型,物资名录ID,标准物资名...",婺城区应急管理局,2024-04-28,2024-06-21,有条件开放,4824,...,['api'],907.0,0.0,0.0,https://open.data.jinhua.gov.cn/jdop_front/det...,每月,1.00,True,2024-04-28,2024.0
428431,NaN,金华市,金华市义乌市“一网通管”行政执法监管-附属客体特种设备信息,市场监督,(市级待审核)xzzf_issue_supervision_fs_equipment-附属客...,义乌市市场监管局,2023-04-24,2024-06-21,有条件开放,4682,...,['api'],905.0,0.0,0.0,https://open.data.jinhua.gov.cn/jdop_front/det...,每年,0.08,True,2023-04-24,2023.0
428432,NaN,金华市,金华市义乌市“一网通管”行政执法监管-附属客体特定产品信息,市场监督,(市级待审核)xzzf_issue_supervision_fs_product-附属客体-...,义乌市市场监管局,2023-04-24,2024-06-21,有条件开放,2355,...,['api'],905.0,0.0,0.0,https://open.data.jinhua.gov.cn/jdop_front/det...,每年,0.08,True,2023-04-24,2023.0


In [47]:
# df_copy.to_csv('./test.csv')